# HuggingFace Hub upload — pipevision yolo26s-seg artifacts

Takes the `model/` folder produced by `sagemaker_seg_train_s.ipynb` Cell 8 and
publishes it to a HuggingFace model repo.

Run this **after** training. Works from a SageMaker notebook (using the
still-staged `model/` folder) or from your local laptop (after you downloaded
the folder).

**You need a HuggingFace write token**: https://huggingface.co/settings/tokens

## 1. Install

In [ ]:
!pip install -q -U 'huggingface_hub>=0.20'

## 2. Configure

- `MODEL_DIR`: folder produced by `sagemaker_seg_train_s.ipynb` Cell 8 (default `model_s`).
- `HF_REPO`: `your-username/repo-name` on HuggingFace.
- `HF_TOKEN`: either set the `HF_TOKEN` env var or paste here.

In [ ]:
import os
from pathlib import Path

MODEL_DIR = 'model_s'                               # folder containing best.pt, onnx, metadata.yaml, ...
HF_REPO   = 'YOUR_USER/pipevision-yolo26s-seg'     # EDIT ME
HF_PRIVATE = False
HF_TOKEN  = os.environ.get('HF_TOKEN', '')         # or paste a token string

MODEL_DIR = Path(MODEL_DIR).resolve()
assert MODEL_DIR.is_dir(), f'Not a directory: {MODEL_DIR}'
assert HF_TOKEN, 'HF_TOKEN missing — set env var or paste a token above.'
assert not HF_REPO.startswith('YOUR_'), 'Set HF_REPO to your-username/repo-name.'

print('model dir:', MODEL_DIR)
print('repo     :', HF_REPO, '(private)' if HF_PRIVATE else '(public)')
print()
print('files to upload:')
for p in sorted(MODEL_DIR.iterdir()):
    if p.is_file():
        print(f'  {p.stat().st_size / 1e6:8.2f} MB  {p.name}')

## 3. Verify the ONNX SHA-256 matches `metadata.yaml`

Catches accidental corruption during download/upload.

In [ ]:
import hashlib, yaml

onnx_file = MODEL_DIR / 'yolo26s-seg-fp16.onnx'
meta_file = MODEL_DIR / 'metadata.yaml'
assert onnx_file.exists(), f'missing {onnx_file}'
assert meta_file.exists(), f'missing {meta_file}'

h = hashlib.sha256()
with open(onnx_file, 'rb') as fh:
    for chunk in iter(lambda: fh.read(1 << 20), b''):
        h.update(chunk)
actual = h.hexdigest()

with open(meta_file) as fh:
    meta = yaml.safe_load(fh)
expected = meta['model']['sha256']

print('actual  :', actual)
print('expected:', expected)
assert actual == expected, 'SHA-256 mismatch — onnx file is corrupt or metadata is stale.'
print('SHA-256 OK')

## 4. Upload to HuggingFace

In [ ]:
from huggingface_hub import HfApi, create_repo

api = HfApi(token=HF_TOKEN)
create_repo(HF_REPO, private=HF_PRIVATE, exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=str(MODEL_DIR),
    repo_id=HF_REPO,
    repo_type='model',
    commit_message=f'upload model (sha256 {actual[:12]})',
)
print(f'Uploaded to https://huggingface.co/{HF_REPO}')
print()
print(f'Set in pipevision-ai/.env.local:')
print(f'  VITE_MODEL_REPO={HF_REPO}')
print(f'  VITE_MODEL_SHA256={actual}')